# LIME for text

- [Online Course](https://www.trainindata.com/p/machine-learning-interpretability)

Let's train NN to predict the topic of conversation of various texts and explain the outputs with LIME.

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np

from sklearn.datasets import fetch_20newsgroups

import keras
from keras import layers
from keras.utils import to_categorical
import tensorflow as tf

from lime.lime_text import LimeTextExplainer

# this is a needed workaround to make LIME work
# with current version of IPython Display
from IPython.display import display, HTML

### Load data

In [2]:
# Selected 4 topic categories from the 
# 20 newsgroups dataset

categories = [
    'alt.atheism',
    'comp.graphics',
    'sci.med',
    'talk.politics.misc',
]

In [3]:
# Load train and test sets

data_train = fetch_20newsgroups(
    subset='train',
    categories=categories,
    shuffle=False,
    remove=('headers', 'footers', 'quotes'))


data_test = fetch_20newsgroups(
    subset='test',
    categories=categories,
    shuffle=False,
    remove=('headers', 'footers', 'quotes'))

In [4]:
# we can find the category names here:

data_train.target_names

['alt.atheism', 'comp.graphics', 'sci.med', 'talk.politics.misc']

In [5]:
# Split text from target

X_train = data_train.data
y_train = data_train.target

X_test = data_test.data
y_test = data_test.target

### Display some texts

Go ahead and change the index of the dataframe to go through different text pieces.

In [6]:
X_train[2]

"\nFlights of fancy, and other irrational approaches, are common.  The crucial\nthing is not to sit around just having fantasies; they aren't of any use\nunless they make you do some experiments.  I've known a lot of scientists\nwhose fantasies lead them on to creative work; usually they won't admit\nout loud what the fantasy was, prior to the consumption of a few beers.\n\n(Simple example: Warren Jelinek noticed an extremely heavy band on a DNA\nelectrophoresis gel of human ALU fragments.  He got very excited, hoping that\nhe'd seen some essential part of the control mechanism for eukaryotic\ngenes.  This fantasy led him to sequence samples of the band and carry out\nbinding assays.  The result was a well-conserved, 400 or so bp, sequence\nthat occurs about 500,000 times in the human genome.  Unfortunately for\nWarren's fantasy, it turns out to be a transposon that is present in\nso many copies because it replicates itself and copies itself back into\nthe genome.  On the other hand, t

In [7]:
# the topic category of the previous text:

y_train[2]

2

### Convert target to matrix

To create a classifier with Keras, the target variable needs to be a matrix, and each class should be referenced in a column:

In [8]:
# This is basically one hot encoding 
# of the target class

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

y_train

array([[1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       ...,
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

## Set up the text classifier

We need a function that converts the word sequence in each text, into a numerical vector. We'll use Keras built-in `VextVectorization` transformer.

In [9]:
# Create text vectorization layer. 

vectorize_layer = layers.TextVectorization(
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    ngrams=None,
    output_mode="int",
    max_tokens=20000,
    output_sequence_length=500,
)

# Now that the vocab layer has been created, call `adapt` on the
# list of strings to create the vocabulary.

vectorize_layer.adapt(X_train)

In [10]:
# test the function on 1 text:

vectorize_layer(X_train[2])

<tf.Tensor: shape=(500,), dtype=int64, numpy=
array([ 5223,     4,  3082,     6,    55,  3068,  2680,    15,   394,
           2,  5917,   193,     7,    17,     3,  2860,   248,    63,
         249,  5234,    27,   722,     4,    47,    83,   562,    27,
          98,    13,    37,    43,  3083,   157,   448,     5,   195,
           4,   973,   910,  5234,  1289,    70,    19,     3,  4786,
         115,   436,    27,   560,  1242,    56,  5135,    33,     2,
        3259,    25,  1615,     3,     2,  4797,     4,     5,   176,
       14190,   465,   202,  6135,     1,  1774,    29,  1626,  1787,
        6028,    19,     5,  2790,     1,  7785,     4,   282, 14384,
       13148,    52,   246,    82,  6685,  3462,     8,  2916,   353,
          43,  1680,   210,     4,     2,   385,  2023,    12, 13323,
        1725,    14,  3259,  1662,   170,     3,  1371,  4538,     4,
           2,  6028,     6,  1505,    56, 14159,     1,     2,   483,
          25,     5, 15232,  3331,    22,   

### Create a 1D convnet to predict topic

We'll create a simple 1D convnet starting with an Embedding layer that can process strings.

In [11]:
# seed for reproducibility
keras.utils.set_random_seed(812)

# map strings to integers
text_input = keras.Input(shape=(1,), dtype=tf.string, name='text')

# convert strings to integers with previous function
x = vectorize_layer(text_input)

# Next, we add a layer to map those vocab indices into a space
# of dimensionality 128:
x = layers.Embedding(20000 + 1, 128)(x)
x = layers.Dropout(0.5)(x)

# Conv1D + global max pooling
x = layers.Conv1D(128, 7, padding="valid", activation="relu", strides=3)(x)
x = layers.Conv1D(128, 7, padding="valid", activation="relu", strides=3)(x)
x = layers.GlobalMaxPooling1D()(x)

# We add a vanilla hidden layer:
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)

# We project onto a 4 unit output layer, a unit per class (topic):
predictions = layers.Dense(4, activation="softmax", name="predictions")(x)

# set up model
model = keras.Model(text_input, predictions)

# Compile the model with categoricaly crossentropy 
# loss and an adam optimizer.
model.compile(loss="categorical_crossentropy",
              optimizer="adam", metrics=["accuracy"])

### Train model

In [12]:
# Fit the model

model.fit(
    x=np.array(X_train, dtype=object),
    y=y_train,
    epochs=5,
)

Epoch 1/5
67/67 ━━━━━━━━━━━━━━━━━━━━ 10s 116ms/step - accuracy: 0.2687 - loss: 1.3862
Epoch 2/5
67/67 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - accuracy: 0.3852 - loss: 1.3436
Epoch 3/5
67/67 ━━━━━━━━━━━━━━━━━━━━ 11s 136ms/step - accuracy: 0.5417 - loss: 1.0319
Epoch 4/5
67/67 ━━━━━━━━━━━━━━━━━━━━ 9s 140ms/step - accuracy: 0.6448 - loss: 0.7721
Epoch 5/5
67/67 ━━━━━━━━━━━━━━━━━━━━ 9s 139ms/step - accuracy: 0.7241 - loss: 0.6087


In [13]:
# evaluate the model on the test set

model.evaluate(np.array(X_test, dtype=object), y_test)

45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.6901 - loss: 0.7258


[0.7589307427406311, 0.6789250373840332]

In [14]:
# get the predictions:

predictions = model.predict(np.array(X_test, dtype=object))

predictions

45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step


array([[5.9128261e-01, 7.5813830e-03, 1.3561114e-02, 3.8757488e-01],
       [5.7832189e-02, 9.1914839e-01, 6.4178761e-03, 1.6601600e-02],
       [3.8516810e-06, 9.9999619e-01, 1.1727844e-10, 1.1842448e-08],
       ...,
       [3.2204425e-06, 9.9999678e-01, 1.2882734e-10, 9.8687511e-09],
       [8.9051951e-05, 9.9990976e-01, 4.3922871e-08, 1.2271998e-06],
       [6.0944587e-01, 3.0470621e-03, 6.1812415e-03, 3.8132578e-01]],
      dtype=float32)

## Interpret with LIME

In [16]:
categories

['alt.atheism', 'comp.graphics', 'sci.med', 'talk.politics.misc']

In [17]:
explainer = LimeTextExplainer(
    class_names=categories,
    random_state=90,
)

## Setting up the predict function

The secret to explaining with LIME is to make the next function work. Unfortunately, there isn't that many examples on how to set up this function properly, so it took me a long time to get it right. Hopefully, you'll do better than me.

This function needs to be able to take in 1 single text in a list, and return a prediction. And it also needs to be able to take several different texts in a list, and output one prediction per text.

It needs 1 text, to return the model prediction for the sample being analyzed.
It needs several texts, because LIME perturbs the sample, and then sends several texts to the model to obtain the predictions that it will then use to fit the Ridge (or any other explainable model).

In [18]:
def predict_fn(text):
    text_to_predict = np.array(text, dtype=object)
    return model.predict(text_to_predict)

predict_fn(["Hola, como estas?"])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


array([[0.2870172 , 0.27421907, 0.19049932, 0.24826446]], dtype=float32)

In [19]:
text = ["Hola, como estas?", "bien y vos?"]

predict_fn(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


array([[0.2870172 , 0.27421907, 0.19049932, 0.24826446],
       [0.28695235, 0.28567407, 0.1843984 , 0.24297513]], dtype=float32)

In [20]:
predict_fn(X_train[0:2])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


array([[6.7795599e-01, 1.7985351e-02, 1.0290586e-02, 2.9376805e-01],
       [1.4976886e-04, 2.7487136e-08, 9.8588330e-01, 1.3966892e-02]],
      dtype=float32)

In [21]:
# test the explainer

exp = explainer.explain_instance("Hola, como estas?", predict_fn, num_features=10)

exp.as_list()

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step


[('Hola', 7.31742799029332e-05),
 ('estas', 7.31590341885158e-05),
 ('como', 7.31335386148934e-05)]

## Explain texts

In [22]:
idx = 5

# the text to explain

X_test[idx]

'\n\n\n \n\nThere can be. But depression is not diagnositic of thyroid deficiency.\nThyroid blood tests are easy, cheap, and effective in diagnosing thyroid\ndeficiencies.'

In [23]:
# the category of the text

y_test[idx]

array([0., 0., 1., 0.])

In [24]:
# the prediction made by the model

predictions[idx].argmax()

2

In [25]:
# obtain the explanation

exp = explainer.explain_instance(X_test[idx], predict_fn, num_features=10, top_labels=4)

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step


In [ ]:
# display explanation for all classes

display(HTML(exp.as_html(text=False)))

In [ ]:
# display explantion for all classes and text

display(HTML(exp.as_html()))

In [ ]:
# display explanation for class of interest and tex

display(HTML(exp.as_html(labels=(0,))))

## Explain another text

In [29]:
idx = 500

X_test[idx]

'------------- cut here -----------------\n \n        ONCE A YEAR...FOR A LIFETIME VIDEO KIT.  This kit\n        includes a 25-minute VHS videotape that presents common\n        misconceptions about mammography.  It tells of the\n        benefits gains by the early detection of breast cancer.\n        Jane Pauley and Phylicia Rashad are the narrators.  Kit\n        includes a guide, poster, flyer, and pamphlets on\n        mammography.  This kit is available directly by writing\n        to:  Modern, 5000 Park Street North, St. Petersburg, FL\n        33709-9989.\n \n \n \nADDITIONAL RESOURCES\n \n \n     COMBINED HEALTH INFORMATION DATABASE (CHID).  A computerized\n     bibliographic database developed and managed by agencies of\n     the U.S. Public Health Service.  It contains references to\n     health information and health education resources.  The\n     database provides bibliographic citations and abstracts for\n     journal articles, books, reports, pamphlets, audiovisuals,\n  

In [30]:
# the category of the text

y_test[idx]

array([0., 0., 1., 0.])

In [31]:
# obtain the explanation

exp = explainer.explain_instance(X_test[idx], predict_fn, num_features=10, top_labels=4)

157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step


In [ ]:
# display explantion for all classes and text

display(HTML(exp.as_html()))